# Models and Materializations

## Overview
A **dbt model** is a `.sql` file containing a single SELECT statement. dbt wraps it in `CREATE VIEW`, `CREATE TABLE`, or similar DDL depending on the **materialization** setting. The analyst writes only the SELECT; dbt handles the rest.

**The four core materializations:**

| Materialization | What dbt creates | Rebuilt on `dbt run`? | Best for |
|---|---|---|---|
| `view` | `CREATE VIEW` | Yes (DDL only, instant) | Staging models; lightweight transforms |
| `table` | `DROP + CREATE TABLE AS SELECT` | Yes (full rebuild) | Mart models; frequently queried results |
| `incremental` | `INSERT` or `MERGE` of new rows only | Only new/changed rows | Large fact tables; event logs |
| `ephemeral` | Compiled as a CTE; never written to DB | N/A | Intermediate logic; not directly queryable |

**Choosing a materialization:**
```
staging layer    →  view       (cheap rebuild; no storage cost)
intermediate     →  ephemeral  (no DB object; compiled inline)
marts / facts    →  table      (persisted; fast for BI tools)
large facts      →  incremental (hours of history; only process new rows)
```

**Config block syntax:**
```sql
{{ config(
    materialized = 'table',
    schema       = 'finance',
    tags         = ['daily', 'finance'],
    indexes      = [{'columns': ['order_id'], 'unique': True}]
) }}
SELECT ...
```

---

In [1]:
stg = """
{{ config(materialized=\"view\") }}
SELECT
    id                                  AS order_id,
    user_id                             AS customer_id,
    status,
    LOWER(TRIM(status))                 AS status_clean,
    amount                              AS amount_cents,
    ordered_at::DATE                    AS order_date,
    _loaded_at                          AS loaded_at
FROM {{ source('jaffle_shop', 'raw_orders') }}
WHERE id IS NOT NULL
"""

eph = """
{{ config(materialized=\"ephemeral\") }}
SELECT
    o.order_id, o.customer_id, o.order_date,
    o.status_clean    AS order_status,
    SUM(p.amount_cents)  AS total_amount_cents,
    COUNT(p.payment_id)  AS payment_count
FROM   {{ ref(\"stg_orders\") }}   AS o
JOIN   {{ ref(\"stg_payments\") }} AS p  ON o.order_id = p.order_id
GROUP BY 1, 2, 3, 4
"""

tbl = """
{{ config(
    materialized = \"table\",
    schema       = \"finance\",
    indexes      = [
        {\"columns\": [\"order_id\"],              \"unique\": True},
        {\"columns\": [\"customer_id\", \"order_date\"]}
    ]
) }}
SELECT
    order_id, customer_id, order_date, order_status,
    total_amount_cents,
    ROUND(total_amount_cents / 100.0, 2) AS total_amount_usd,
    payment_count,
    CURRENT_TIMESTAMP                    AS dbt_updated_at
FROM {{ ref(\"int_order_payments_joined\") }}
"""

print("--- staging/stg_orders.sql (view) ---")
print(stg)
print("--- intermediate/int_order_payments_joined.sql (ephemeral) ---")
print(eph)
print("--- marts/finance/fct_orders.sql (table) ---")
print(tbl)

--- staging/stg_orders.sql (view) ---

{{ config(materialized="view") }}
SELECT
    id                                  AS order_id,
    user_id                             AS customer_id,
    status,
    LOWER(TRIM(status))                 AS status_clean,
    amount                              AS amount_cents,
    ordered_at::DATE                    AS order_date,
    _loaded_at                          AS loaded_at
FROM {{ source('jaffle_shop', 'raw_orders') }}
WHERE id IS NOT NULL

--- intermediate/int_order_payments_joined.sql (ephemeral) ---

{{ config(materialized="ephemeral") }}
SELECT
    o.order_id, o.customer_id, o.order_date,
    o.status_clean    AS order_status,
    SUM(p.amount_cents)  AS total_amount_cents,
    COUNT(p.payment_id)  AS payment_count
FROM   {{ ref("stg_orders") }}   AS o
JOIN   {{ ref("stg_payments") }} AS p  ON o.order_id = p.order_id
GROUP BY 1, 2, 3, 4

--- marts/finance/fct_orders.sql (table) ---

{{ config(
    materialized = "table",
    schema    

---
## How dbt compiles models to SQL

In [2]:
compiled_view = """
CREATE VIEW analytics.staging.stg_orders AS (
  SELECT id AS order_id, user_id AS customer_id, status,
         LOWER(TRIM(status)) AS status_clean, amount AS amount_cents,
         ordered_at::DATE AS order_date, _loaded_at AS loaded_at
  FROM raw.jaffle_shop.raw_orders
  WHERE id IS NOT NULL
);
"""

compiled_table = """
DROP TABLE IF EXISTS analytics.finance.fct_orders CASCADE;

CREATE TABLE analytics.finance.fct_orders AS (
  WITH int_order_payments_joined AS (   -- ephemeral model inlined as CTE
      SELECT o.order_id, o.customer_id, o.order_date,
             o.status_clean AS order_status,
             SUM(p.amount_cents) AS total_amount_cents,
             COUNT(p.payment_id) AS payment_count
      FROM   analytics.staging.stg_orders   AS o
      JOIN   analytics.staging.stg_payments AS p ON o.order_id = p.order_id
      GROUP BY 1, 2, 3, 4
  )
  SELECT order_id, customer_id, order_date, order_status,
         total_amount_cents,
         ROUND(total_amount_cents / 100.0, 2) AS total_amount_usd,
         payment_count, CURRENT_TIMESTAMP AS dbt_updated_at
  FROM int_order_payments_joined
);

CREATE UNIQUE INDEX ON analytics.finance.fct_orders (order_id);
CREATE INDEX ON analytics.finance.fct_orders (customer_id, order_date);
"""

print("=== Compiled SQL: stg_orders (view) ===")
print(compiled_view)
print("=== Compiled SQL: fct_orders (table, with inlined ephemeral CTE) ===")
print(compiled_table)

=== Compiled SQL: stg_orders (view) ===

CREATE VIEW analytics.staging.stg_orders AS (
  SELECT id AS order_id, user_id AS customer_id, status,
         LOWER(TRIM(status)) AS status_clean, amount AS amount_cents,
         ordered_at::DATE AS order_date, _loaded_at AS loaded_at
  FROM raw.jaffle_shop.raw_orders
  WHERE id IS NOT NULL
);

=== Compiled SQL: fct_orders (table, with inlined ephemeral CTE) ===

DROP TABLE IF EXISTS analytics.finance.fct_orders CASCADE;

CREATE TABLE analytics.finance.fct_orders AS (
  WITH int_order_payments_joined AS (   -- ephemeral model inlined as CTE
      SELECT o.order_id, o.customer_id, o.order_date,
             o.status_clean AS order_status,
             SUM(p.amount_cents) AS total_amount_cents,
             COUNT(p.payment_id) AS payment_count
      FROM   analytics.staging.stg_orders   AS o
      JOIN   analytics.staging.stg_payments AS p ON o.order_id = p.order_id
      GROUP BY 1, 2, 3, 4
  )
  SELECT order_id, customer_id, order_date, order

---
## Materialization decision matrix

In [3]:
rows = [
    ("stg_customers",        "view",        "Instant DDL; rebuilt every run; fine for small sources"),
    ("int_customer_orders",  "ephemeral",   "Inlined as CTE; saves a DB round-trip; not queryable directly"),
    ("dim_customers",        "table",       "BI tools query it directly; persistent; fast reads"),
    ("fct_page_views",       "incremental", "Billions of rows; only append today rows; hours of data history"),
    ("dim_dates",            "seed",        "Static calendar table; loaded from CSV; rarely changes"),
    ("fct_daily_summary",    "table",       "Aggregated from fct_page_views; small enough to rebuild daily"),
]
print("=== Materialization decision matrix ===")
print(f"  {'Model':<28s}  {'Materialization':<14s}  Rationale")
print("  " + "-"*75)
for model, mat, reason in rows:
    print(f"  {model:<28s}  {mat:<14s}  {reason}")
print()
print("=== Viewing model metadata after dbt run ===")
print("""
  -- In your PostgreSQL analytics schema after dbt run:
  SELECT table_name, table_type
  FROM   information_schema.tables
  WHERE  table_schema = 'staging'
  ORDER BY table_type, table_name;
  -- Views show as VIEW, tables as BASE TABLE.
  -- Ephemeral models do NOT appear (no DB object).
""")

=== Materialization decision matrix ===
  Model                         Materialization  Rationale
  ---------------------------------------------------------------------------
  stg_customers                 view            Instant DDL; rebuilt every run; fine for small sources
  int_customer_orders           ephemeral       Inlined as CTE; saves a DB round-trip; not queryable directly
  dim_customers                 table           BI tools query it directly; persistent; fast reads
  fct_page_views                incremental     Billions of rows; only append today rows; hours of data history
  dim_dates                     seed            Static calendar table; loaded from CSV; rarely changes
  fct_daily_summary             table           Aggregated from fct_page_views; small enough to rebuild daily

=== Viewing model metadata after dbt run ===

  -- In your PostgreSQL analytics schema after dbt run:
  SELECT table_name, table_type
  FROM   information_schema.tables
  WHERE  table_s

---
## Common Pitfalls

**1. Using `table` materialization for staging models**
Staging models are rebuilt every `dbt run` anyway, so materialising them as tables wastes storage and lengthens run time. `view` is almost always correct for staging: rebuild is instant (DDL only), and the underlying source is always fresh.

**2. Using `view` for large mart models queried by BI tools**
A view re-runs the full SELECT every time a dashboard loads. A mart with 10 joins and 50 million rows as a view means every dashboard refresh executes a multi-second query. Materialise heavily-queried marts as `table` so BI tools read pre-computed results.

**3. Referencing ephemeral models directly in SQL or BI tools**
Ephemeral models have no database object. `SELECT * FROM analytics.staging.int_order_payments_joined` fails because the table does not exist. Only other dbt models can `ref()` an ephemeral model -- it is inlined as a CTE at compile time. If you need to query the result directly, change it to `view`.

**4. Forgetting `--full-refresh` after changing an incremental model's column list**
`table` models drop and recreate on every run, so column changes are picked up automatically. `incremental` models do NOT drop and recreate -- they merge into an existing table. If you add a column, run `dbt run --full-refresh -s <model>` once to rebuild the table with the new schema.

**5. Inconsistent materialization overrides between config block and project defaults**
If `dbt_project.yml` sets `+materialized: view` for staging but an individual model has `{ config(materialized='table') }`, that model is a table -- even in the staging folder. This is often unintentional. Prefer project-level defaults for consistency; use config blocks only for deliberate per-model exceptions.

**6. Not declaring indexes on `table` materializations in PostgreSQL**
dbt creates tables without any indexes by default. A `fct_orders` table with no index on `customer_id` does a full seq scan on every BI query. Declare indexes in the config block using the `indexes` list.


---
*sql_methods_library - Samantha McGarrigle*